In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA07 - Marketing (Canada, Hong Kong, Barbados Only)
   
   BUSINESS QUESTION:
   Did the AU incur marketing expenses during the assessment period?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. FINANCE DATA ONLY: Unions the 6 Finance files. Coupa data is excluded.
   3. CATEGORY CODE FILTER: Strictly filters the Finance data for Marketing codes 
      ('194', '370', '372', '377', '408', '628', '629', '653', '830', '875', '877').
   4. MAPPING AGGREGATION: Joins the Cost Center Mapping Bootstrap to the transactions 
      to determine which AUs had marketing spend. Evaluates ALL mapped AUs.
   5. FINAL OUTPUT: LEFT JOINS the mapping results onto the Master List anchor.
      - Outputs 'Yes' if marketing spend exists.
      - Outputs 'No' if the AU maps to cost centers but has no marketing spend.
      - Outputs '' (Blank) if the AU mapping is missing entirely.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Finance_Raw AS (
    -- 2. Union all 6 Finance files into one master dataset
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Marketing_Transactions AS (
    -- 3. Filter STRICTLY for marketing category codes
    SELECT 
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRIM(CatCode) AS CatCode
    FROM Combined_Finance_Raw
    WHERE TRIM(CatCode) IN (
        '194', '370', '372', '377', '408', '628', 
        '629', '653', '830', '875', '877'
    )
      AND CostCenter IS NOT NULL
),

CC_Mapping AS (
    -- Standardize the CC Mapping table using the correct cost_center_id column
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

AU_Marketing_Flags AS (
    -- 4. Map transactions to ALL AUs in the bootstrap and flag if they have marketing spend
    SELECT 
        m.AU_ID,
        MAX(CASE WHEN mt.Cleaned_CC IS NOT NULL THEN 1 ELSE 0 END) AS Has_Marketing
    FROM CC_Mapping m
    LEFT JOIN Marketing_Transactions mt
        ON m.Mapped_CC = mt.Cleaned_CC
    GROUP BY m.AU_ID
)

-- 5. Build final report using the Filtered Master List as the anchor
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EBA07' AS QuestionID,               
    
    CASE 
        WHEN amf.Has_Marketing = 1 THEN 'Yes' 
        -- It exists in the mapping table but has NO marketing spend
        WHEN amf.AU_ID IS NOT NULL THEN 'No' 
        -- It ONLY exists in the Master List (missing mapping/data entirely)
        ELSE 'No' 
    END AS Response,
    
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN AU_Marketing_Flags amf
    ON a.BusinessID = amf.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA07 Drill-Down
   
   PURPOSE: Shows EVERY marketing transaction from Finance, regardless of whether 
   the Cost Center maps to an AU, or whether that AU exists in the Master List. 
   Useful for tracking unmapped marketing expenses.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Finance_Parsed AS (
    -- Extract the Raw String and parsed elements, filtered strictly for Marketing
    SELECT 
        CostCenter AS Raw_String,
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRIM(CatCode) AS CatCode,
        'Finance' AS Source_System
    FROM hive_metastore.ra_adido_2025.f25_finance_data_1 WHERE TRIM(CatCode) IN ('194', '370', '372', '377', '408', '628', '629', '653', '830', '875', '877')
    UNION ALL SELECT CostCenter, CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END, TRIM(CatCode), 'Finance' FROM hive_metastore.ra_adido_2025.f25_finance_data_2 WHERE TRIM(CatCode) IN ('194', '370', '372', '377', '408', '628', '629', '653', '830', '875', '877')
    UNION ALL SELECT CostCenter, CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END, TRIM(CatCode), 'Finance' FROM hive_metastore.ra_adido_2025.f25_finance_data_3 WHERE TRIM(CatCode) IN ('194', '370', '372', '377', '408', '628', '629', '653', '830', '875', '877')
    UNION ALL SELECT CostCenter, CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END, TRIM(CatCode), 'Finance' FROM hive_metastore.ra_adido_2025.f25_finance_data_4 WHERE TRIM(CatCode) IN ('194', '370', '372', '377', '408', '628', '629', '653', '830', '875', '877')
    UNION ALL SELECT CostCenter, CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END, TRIM(CatCode), 'Finance' FROM hive_metastore.ra_adido_2025.f25_finance_data_5 WHERE TRIM(CatCode) IN ('194', '370', '372', '377', '408', '628', '629', '653', '830', '875', '877')
    UNION ALL SELECT CostCenter, CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END, TRIM(CatCode), 'Finance' FROM hive_metastore.ra_adido_2025.f25_finance_data_6 WHERE TRIM(CatCode) IN ('194', '370', '372', '377', '408', '628', '629', '653', '830', '875', '877')
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
)

SELECT 
    COALESCE(m.AU_ID, 'UNMAPPED_CC') AS BusinessID,
    COALESCE(mast.In_ABAC_AU_List, 'No') AS In_ABAC_AU_List,
    f.Cleaned_CC AS Flagged_Cost_Center,
    f.CatCode AS Marketing_Category_Code,
    f.Source_System,
    f.Raw_String AS Original_Data_Value
FROM Finance_Parsed f
LEFT JOIN CC_Mapping m
    ON f.Cleaned_CC = m.Mapped_CC
LEFT JOIN Master_AUs mast
    ON m.AU_ID = mast.BusinessID
WHERE f.Cleaned_CC IS NOT NULL
ORDER BY BusinessID, f.Cleaned_CC;

In [ ]:
%sql
/* ===================================================================================
   QA DIFF LOGIC: Missing AUs for Canada, Barbados, Hong Kong (EBA07)
   
   PURPOSE: 
   Identifies Business IDs that exist in abac_2025.fy25_list_of_unit for specific 
   jurisdictions but have NO matching marketing transactions in the EBA07 logic.
=================================================================================== */

WITH Filtered_Master_List AS (
    -- 1. Grab the Master AUs and filter jurisdiction for the target countries
    SELECT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name,
        TRIM(jurisdiction) AS jurisdiction
    FROM abac_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
      AND TRIM(jurisdiction) IN ('Canada', 'Hong Kong', 'Barbados')
),

Combined_Finance_Raw AS (
    -- 2. Pull the 6 Finance files based on your EBA07 logic
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Marketing_Transactions AS (
    -- 3. Filter for Marketing Category Codes
    SELECT 
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC
    FROM Combined_Finance_Raw
    WHERE TRIM(CatCode) IN (
        '194', '370', '372', '377', '408', '628', 
        '629', '653', '830', '875', '877'
    )
),

EBA07_Successful_AUs AS (
    -- 4. Get the distinct Business IDs that successfully matched to a marketing transaction
    SELECT DISTINCT 
        TRIM(CAST(m.AU_ID AS STRING)) AS BusinessID
    FROM vw_cost_center_mapping_bootstrap m
    INNER JOIN Marketing_Transactions mt
        ON CASE WHEN TRIM(m.Cost_Center_ID) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(m.Cost_Center_ID), 4), 4, '0') ELSE UPPER(TRIM(m.Cost_Center_ID)) END = mt.Cleaned_CC
)

-- 5. Find the Difference (What is in the Filtered Master List but NOT in EBA07 Successful AUs)
SELECT 
    fml.BusinessID AS Missing_BusinessID,
    fml.AU_Name,
    fml.jurisdiction,
    'No EBA07 Marketing Transactions Mapped' AS Diff_Reason
    
FROM Filtered_Master_List fml

-- LEFT JOIN targeting NULLs isolates the rows that are missing
LEFT JOIN EBA07_Successful_AUs eba 
    ON fml.BusinessID = eba.BusinessID
WHERE eba.BusinessID IS NULL

ORDER BY fml.BusinessID;